# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Adjust this path if your repo is stored elsewhere in Drive.
# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [ ]:
# # Install Python dependencies (run once per session)
# !pip install -r {PROJECT_ROOT}/requirements.txt -q
# !python -m spacy download en

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/COMP5329_Assignment1-main
!ls
import os
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/COMP5329_Assignment1-main
assignment1.ipynb  Losses	       requirements.txt
_data		   _model	       Schedulers
Data		   _model_dropout_off  STAGE12_CODE_CHANGES_BY_MODULE.md
EvaluateTools	   _model_dropout_on   STAGE1_DEBUG_LOG.md
_log		   _model_group_norm   STAGE2_DEBUG_LOG.md
_log_dropout_off   _model_layer_norm   Tools
_log_dropout_on    Models	       TrainTools
_log_group_norm    Optimizers
_log_layer_norm    README.md
/content/drive/MyDrive/COMP5329_Assignment1-main


In [2]:
#  Install Python dependencies (run once per session)
!pip install -r requirements.txt
!python -m spacy download en

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 2.6 MB/s eta 0:00:00
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 198.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [3]:
import sys, os
# local root
from pathlib import Path
PROJECT_ROOT = str(Path.cwd().resolve())

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: /content/drive/MyDrive/COMP5329_Assignment1-main


---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [4]:
from Tools.download import download_mini

download_mini(data_dir="_data")

Step 1 / 2  —  Mini dataset (SQuAD + GloVe)
  [skip] Mini dataset already present in _data/.

Step 2 / 2  —  spaCy language model
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

Mini dataset download complete.


---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [5]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

Generating train examples…


100%|██████████| 150/150 [00:02<00:00, 58.12it/s]


  30293 questions in total
Generating dev examples…


100%|██████████| 48/48 [00:00<00:00, 55.22it/s]


  10570 questions in total
Generating word embedding…


...

100%|██████████| 30293/30293 [00:02<00:00, 10688.00it/s]


  Built 30169 / 30293 instances
Processing dev examples…


100%|██████████| 10570/10570 [00:01<00:00, 9827.80it/s]


  Built 10465 / 10570 instances
Saving word embedding…
Saving char embedding…
Saving train eval…
Saving dev eval…
Saving word dictionary…
Saving char dictionary…
Saving dev meta…

Preprocessing complete.
  Outputs → _data/


{'train_record_file': '_data/train.npz',
 'dev_record_file': '_data/dev.npz',
 'word_emb_file': '_data/word_emb.json',
 'char_emb_file': '_data/char_emb.json',
 'train_eval_file': '_data/train_eval.json',
 'dev_eval_file': '_data/dev_eval.json',
 'word2idx_file': '_data/word2idx.json',
 'char2idx_file': '_data/char2idx.json',
 'dev_meta_file': '_data/dev_meta.json'}

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [6]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 1000,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "none",
    loss_name      = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 200/200 [00:12<00:00, 15.55it/s]


STEP      200  loss 1826.203199



100%|██████████| 150/150 [00:02<00:00, 62.86it/s]


VALID(train) loss 34.374232  F1 7.010705  EM 0.000000



100%|██████████| 150/150 [00:02<00:00, 64.04it/s]


...

STEP     1000  loss 210.450797



100%|██████████| 150/150 [00:02<00:00, 64.71it/s]


VALID(train) loss 16.727654  F1 6.726962  EM 0.416667



100%|██████████| 150/150 [00:02<00:00, 64.36it/s]


TEST        loss 17.157789  F1 5.808303  EM 0.166667

Learning rate: [0.001]
Training finished.  Best F1: 6.2431  Best EM: 0.1667
Best F1: 6.2431  |  Best EM: 0.1667


Experiment: LayerNorm(A) vs GroupNorm(B)

Experiment A

In [8]:
from TrainTools.train import train

results_layer_norm = train(
    # data paths
    train_npz        = "_data/train.npz",
    dev_npz          = "_data/dev.npz",
    word_emb_json    = "_data/word_emb.json",
    char_emb_json    = "_data/char_emb.json",
    train_eval_json  = "_data/train_eval.json",
    dev_eval_json    = "_data/dev_eval.json",
    save_dir         = "_model_layer_norm",
    log_dir          = "_log_layer_norm",
    ckpt_name        = "model.pt",

    # training loop
    batch_size       = 8,
    num_steps        = 1000,
    checkpoint       = 200,
    val_num_batches  = 150,
    test_num_batches = 150,
    seed             = 42,

    # optimization
    optimizer_name   = "sgd",
    scheduler_name   = "none",
    loss_name        = "qa_nll",

    # normalization experiment
    norm_name        = "layer_norm",
    norm_groups      = 8,
)

print(f"LayerNorm | Best F1: {results_layer_norm['best_f1']:.4f} | Best EM: {results_layer_norm['best_em']:.4f}")

100%|██████████| 200/200 [00:12<00:00, 15.53it/s]


STEP      200  loss 1826.203199



100%|██████████| 150/150 [00:02<00:00, 63.93it/s]


VALID(train) loss 34.374232  F1 7.010705  EM 0.000000



100%|██████████| 150/150 [00:02<00:00, 64.04it/s]


...

STEP     1000  loss 210.450797



100%|██████████| 150/150 [00:02<00:00, 64.20it/s]


VALID(train) loss 16.727654  F1 6.726962  EM 0.416667



100%|██████████| 150/150 [00:02<00:00, 64.54it/s]


TEST        loss 17.157789  F1 5.808303  EM 0.166667

Learning rate: [0.001]
Training finished.  Best F1: 6.2431  Best EM: 0.1667
LayerNorm | Best F1: 6.2431 | Best EM: 0.1667


Experiment B

In [9]:
from TrainTools.train import train

results_group_norm = train(
    # data paths
    train_npz        = "_data/train.npz",
    dev_npz          = "_data/dev.npz",
    word_emb_json    = "_data/word_emb.json",
    char_emb_json    = "_data/char_emb.json",
    train_eval_json  = "_data/train_eval.json",
    dev_eval_json    = "_data/dev_eval.json",
    save_dir         = "_model_group_norm",
    log_dir          = "_log_group_norm",
    ckpt_name        = "model.pt",

    # training loop
    batch_size       = 8,
    num_steps        = 1000,
    checkpoint       = 200,
    val_num_batches  = 150,
    test_num_batches = 150,
    seed             = 42,

    # optimization
    optimizer_name   = "sgd",
    scheduler_name   = "none",
    loss_name        = "qa_nll",

    # normalization experiment
    norm_name        = "group_norm",
    norm_groups      = 8,
)

print(f"GroupNorm | Best F1: {results_group_norm['best_f1']:.4f} | Best EM: {results_group_norm['best_em']:.4f}")

100%|██████████| 200/200 [00:14<00:00, 14.16it/s]


STEP      200  loss 2316.480887



100%|██████████| 150/150 [00:03<00:00, 49.36it/s]


VALID(train) loss 40.567155  F1 6.918415  EM 0.083333



100%|██████████| 150/150 [00:03<00:00, 49.39it/s]


...

STEP     1000  loss 280.859886



100%|██████████| 150/150 [00:03<00:00, 49.29it/s]


VALID(train) loss 19.641781  F1 6.296202  EM 0.083333



100%|██████████| 150/150 [00:03<00:00, 49.38it/s]


TEST        loss 20.462778  F1 5.182666  EM 0.000000

Learning rate: [0.001]
Training finished.  Best F1: 5.8293  Best EM: 0.1667
GroupNorm | Best F1: 5.8293 | Best EM: 0.1667


In [10]:
print("===== Training Summary =====")
print(f"LayerNorm | Best F1: {results_layer_norm['best_f1']:.4f} | Best EM: {results_layer_norm['best_em']:.4f}")
print(f"GroupNorm | Best F1: {results_group_norm['best_f1']:.4f} | Best EM: {results_group_norm['best_em']:.4f}")

===== Training Summary =====
LayerNorm | Best F1: 6.2431 | Best EM: 0.1667
GroupNorm | Best F1: 5.8293 | Best EM: 0.1667


---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [11]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

100%|██████████| 1309/1309 [00:20<00:00, 64.07it/s]


TEST  loss 17.696156  F1 7.677414  EM 0.210225
F1: 7.6774  |  EM: 0.2102  |  Loss: 17.696156


Custom evaluation function

During evaluation, we encountered a parameter mismatch issue when loading checkpoints trained with different normalization strategies. This occurred because the original evaluation function did not support configurable normalization methods and defaulted to LayerNorm.

To resolve this, we implemented a custom evaluation function that explicitly passes the normalization configuration (e.g., GroupNorm) when reconstructing the model. This ensures consistency between training and evaluation, allowing the model parameters to be loaded correctly.

In [12]:
import os
import argparse
import torch
import ujson as json

from Data import SQuADDataset, load_dev_eval, load_word_char_mats
from Losses import losses
from Models import QANet
from EvaluateTools.eval_utils import run_eval

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def evaluate_norm(
    dev_npz="_data/dev.npz",
    word_emb_json="_data/word_emb.json",
    char_emb_json="_data/char_emb.json",
    dev_eval_json="_data/dev_eval.json",
    save_dir="_model",
    log_dir="_log",
    ckpt_name="model.pt",
    batch_size=8,
    test_num_batches=-1,
    loss_name="qa_nll",
    para_limit=400,
    ques_limit=50,
    char_limit=16,
    d_model=96,
    num_heads=8,
    glove_dim=300,
    char_dim=64,
    dropout=0.1,
    dropout_char=0.05,
    pretrained_char=False,
    norm_name="layer_norm",
    norm_groups=8,
    activation="relu",
    init_name="kaiming",
):
    os.makedirs(log_dir, exist_ok=True)

    args = argparse.Namespace(
        dev_npz=dev_npz,
        word_emb_json=word_emb_json,
        char_emb_json=char_emb_json,
        dev_eval_json=dev_eval_json,
        para_limit=para_limit,
        ques_limit=ques_limit,
        char_limit=char_limit,
        d_model=d_model,
        num_heads=num_heads,
        glove_dim=glove_dim,
        char_dim=char_dim,
        dropout=dropout,
        dropout_char=dropout_char,
        pretrained_char=pretrained_char,
        norm_name=norm_name,
        norm_groups=norm_groups,
        activation=activation,
        init_name=init_name,
    )

    word_mat, char_mat = load_word_char_mats(args)
    model = QANet(word_mat, char_mat, args).to(DEVICE)

    dev_eval = load_dev_eval(args)
    dev_dataset = SQuADDataset(dev_npz)

    ckpt_path = os.path.join(save_dir, ckpt_name)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])

    metrics, ans = run_eval(
        model,
        dev_dataset,
        dev_eval,
        num_batches=test_num_batches,
        batch_size=batch_size,
        use_random_batches=False,
        device=DEVICE,
        loss_fn=losses[loss_name],
    )

    with open(os.path.join(log_dir, "answers.json"), "w") as f:
        json.dump(ans, f)

    print("TEST  loss {loss:.6f}  F1 {f1:.6f}  EM {exact_match:.6f}".format(**metrics))
    return {
        "f1": metrics["f1"],
        "exact_match": metrics["exact_match"],
        "loss": metrics["loss"],
    }

Evaluate A

In [13]:
metrics_layer_norm = evaluate_norm(
    dev_npz        = "_data/dev.npz",
    word_emb_json  = "_data/word_emb.json",
    char_emb_json  = "_data/char_emb.json",
    dev_eval_json  = "_data/dev_eval.json",
    save_dir       = "_model_layer_norm",
    log_dir        = "_log_layer_norm",
    ckpt_name      = "model.pt",
    norm_name      = "layer_norm",
    norm_groups    = 8,
)

print(f"LayerNorm | F1: {metrics_layer_norm['f1']:.4f} | EM: {metrics_layer_norm['exact_match']:.4f} | Loss: {metrics_layer_norm['loss']:.4f}")

100%|██████████| 1309/1309 [00:20<00:00, 64.07it/s]


TEST  loss 17.696156  F1 7.677414  EM 0.210225
LayerNorm | F1: 7.6774 | EM: 0.2102 | Loss: 17.6962


Evaluate B

In [14]:
metrics_group_norm = evaluate_norm(
    dev_npz        = "_data/dev.npz",
    word_emb_json  = "_data/word_emb.json",
    char_emb_json  = "_data/char_emb.json",
    dev_eval_json  = "_data/dev_eval.json",
    save_dir       = "_model_group_norm",
    log_dir        = "_log_group_norm",
    ckpt_name      = "model.pt",
    norm_name      = "group_norm",
    norm_groups    = 8,
)

print(f"GroupNorm | F1: {metrics_group_norm['f1']:.4f} | EM: {metrics_group_norm['exact_match']:.4f} | Loss: {metrics_group_norm['loss']:.4f}")

100%|██████████| 1309/1309 [00:26<00:00, 49.46it/s]


TEST  loss 21.284162  F1 7.601492  EM 0.047778
GroupNorm | F1: 7.6015 | EM: 0.0478 | Loss: 21.2842


In [15]:
print("===== Final Comparison =====")
print(f"LayerNorm | Train Best F1: {results_layer_norm['best_f1']:.4f} | Train Best EM: {results_layer_norm['best_em']:.4f} | Eval F1: {metrics_layer_norm['f1']:.4f} | Eval EM: {metrics_layer_norm['exact_match']:.4f}")
print(f"GroupNorm | Train Best F1: {results_group_norm['best_f1']:.4f} | Train Best EM: {results_group_norm['best_em']:.4f} | Eval F1: {metrics_group_norm['f1']:.4f} | Eval EM: {metrics_group_norm['exact_match']:.4f}")

===== Final Comparison =====
LayerNorm | Train Best F1: 6.2431 | Train Best EM: 0.1667 | Eval F1: 7.6774 | Eval EM: 0.2102
GroupNorm | Train Best F1: 5.8293 | Train Best EM: 0.1667 | Eval F1: 7.6015 | Eval EM: 0.0478


# Experiment: Effect of Normalization Strategy on QANet

## 1. Research Question  
How does the choice of normalization method (LayerNorm vs GroupNorm) affect model performance and generalization in the repaired QANet?

---

## 2. Hypothesis  
We hypothesize that **LayerNorm** will outperform **GroupNorm**, as it is more suitable for sequence-based architectures and is widely used in transformer-style models.

---

## 3. Experimental Setup  

We conduct a controlled experiment by modifying only the normalization strategy while keeping all other hyperparameters fixed.

- Model: repaired QANet  
- Dataset: SQuAD v1.1  
- Batch size: 8  
- Training steps: 1000  
- Optimizer: SGD  
- Scheduler: none  
- Loss: QA NLL  
- Seed: 42  

Two configurations were evaluated:

- **LayerNorm**
- **GroupNorm (8 groups)**

To ensure consistency between training and evaluation, a **custom evaluation function** was implemented so that the normalization configuration used during evaluation matches the training setup.

---

## 4. Results  

| Setting     | Train F1 | Train EM | Eval F1 | Eval EM |
|------------|---------|---------|--------|--------|
| LayerNorm  | 6.2431  | 0.1667  | **7.6774** | **0.2102** |
| GroupNorm  | 5.8293  | 0.1667  | 7.6015 | 0.0478 |

---

## 5. Analysis  

The results show that **LayerNorm consistently outperforms GroupNorm** across both training and evaluation metrics. In particular, LayerNorm achieves higher F1 and EM on the evaluation set, indicating better answer quality and more accurate span prediction.

Compared to shorter training (e.g., 400 steps), the performance gap becomes more evident when the number of training steps increases to 1000. This suggests that the choice of normalization strategy has a stronger impact as the model continues training.

One possible explanation is that LayerNorm operates across feature dimensions within each token, making it well-suited for sequence modeling and attention-based architectures such as QANet. In contrast, GroupNorm normalizes across channel groups, which may not align well with the model structure, leading to less stable training and poorer generalization.

Interestingly, GroupNorm’s Exact Match (EM) decreases with longer training, suggesting that its predictions become less consistent over time. This further indicates weaker generalization under this configuration.

Overall, the results demonstrate that normalization strategy plays an important role in model performance, and **LayerNorm is a more effective choice for this task**.

---

## 6. Conclusion  

Increasing training steps reveals that **LayerNorm not only provides better training stability but also significantly improves generalization compared to GroupNorm**.

---

## 7. Limitations and Future Work  

This experiment is conducted with a fixed set of hyperparameters and a single random seed. Future work could explore:

- Multiple random seeds for robustness  
- Different training durations  
- Other normalization methods (e.g., BatchNorm)  
- Interaction with other components (e.g., dropout, optimizer)